# Data coverage and lagged form audit

Reconstructed midterm companion, executed now. Original source CSV and scraper are preserved separately. These outputs verify reproducibility; they do not backdate work.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load_matches, prepare_binary, dataset_audit, chronological_split, BASE_FEATURES
from src.features import add_rolling_features, verify_rolling_features, ROLLING_FEATURES
from src.experiments import models, evaluate


In [2]:
raw = load_matches()
print(json.dumps(dataset_audit(raw), indent=2))

{
  "rows": 1389,
  "columns_without_saved_index": 27,
  "start": "2020-09-12",
  "end": "2022-04-25",
  "season_rows": {
    "2021": 760,
    "2022": 629
  },
  "season_clubs": {
    "2021": 20,
    "2022": 19
  },
  "outcomes": {
    "L": 548,
    "W": 526,
    "D": 315
  },
  "duplicate_team_date_opponent": 0,
  "missing_2021_22_liverpool": true
}


## Explicit historical split

Strict inequalities reproduce the recorded binary experiment. The cutoff-day records are excluded. This is not the planned four-way three-class protocol.

In [3]:
train, test = chronological_split(raw)
print(f"Train: {len(train)} rows, through {train.date.max().date()}")
print(f"Test: {len(test)} rows, from {test.date.min().date()}")
print("Excluded cutoff rows:", int(raw.date.eq("2022-01-01").sum()))

Train: 1107 rows, through 2021-12-30
Test: 276 rows, from 2022-01-02
Excluded cutoff rows: 6


## Rolling features: prior appearances only

Each value is checked against the previous three rows of the same team. The Manchester City example must equal 10/3 goals for and 1/3 goals against.

In [4]:
rolling = add_rolling_features(raw)
print(json.dumps(verify_rolling_features(rolling), indent=2))
example = rolling[(rolling.team == "Manchester City") & rolling.date.eq("2021-09-11")]
print(example[["date", "team", "gf_rolling", "ga_rolling"]].to_string(index=False))

{
  "checks_passed": 10560,
  "complete_rows": 1317
}
      date            team  gf_rolling  ga_rolling
2021-09-11 Manchester City    3.333333    0.333333


## Remaining work

Audit one record per fixture, fix missing coverage, build H/D/A targets, and freeze training/tuning/calibration/final-test partitions before the final research comparison.